In [8]:
import shutil
from os import path
import os
import json

# Define the folder locations of source and destination datasets
SRC_DIR = '/Users/sanikajumde/Downloads/archive (3)'  # Source dataset folder
DEST_DIR = '/Users/sanikajumde/Downloads/yolo_dataset'  # Destination dataset folder

# Step 1: Create folder structure of destination dataset
os.makedirs(path.join(DEST_DIR, "train", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "train", "labels"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "val", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "val", "labels"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "test", "images"), exist_ok=True)
os.makedirs(path.join(DEST_DIR, "test", "labels"), exist_ok=True)

# Step 2: Generate the data.yaml file
meta = json.load(open(path.join(SRC_DIR, "meta.json")))
classes = {}
for index, entry in enumerate(meta["classes"]):
    classes[entry["title"]] = index

# Create the "data.yaml" file in the destination dataset
with open(path.join(DEST_DIR, "data.yaml"), "w") as fp:
    fp.write("train: ../train/images\n")
    fp.write("val: ../val/images\n")
    fp.write("test: ../test/images\n")
    fp.write("\n")
    fp.write("nc: {}\n".format(len(classes)))
    fp.write("names: ['{}']\n".format("','".join(classes.keys())))

# Step 3: Copy images from source to destination datasets
dirs_map = {"train": "train", "valid": "val", "test": "test"}

for src_dir, dest_dir in dirs_map.items():
    # Copy all images from source to destination dataset
    shutil.copytree(path.join(SRC_DIR, src_dir, "img"),
                    path.join(DEST_DIR, dest_dir, "images"),
                    dirs_exist_ok=True)

    # Step 4: Convert annotations
    for file in os.listdir(path.join(SRC_DIR, src_dir, "ann")):
        ann = json.load(open(path.join(SRC_DIR, src_dir, "ann", file), "r"))
        
        # Get width and height of the image
        img_width = ann["size"]["width"]
        img_height = ann["size"]["height"]
        
        # Prepare name for the destination annotation file
        file_name = file.replace(".jpg.json", ".txt")
        with open(path.join(DEST_DIR, dest_dir, "labels", file_name), "w") as fp:
            
            for obj in ann["objects"]:
                class_id = classes[obj["classTitle"]]
                
                # Initialize bounding box coordinates
                top = float('inf')
                left = float('inf')
                bottom = float('-inf')
                right = float('-inf')

                # Calculate bounding box from polygon points
                for point in obj["points"]["exterior"]:
                    x, y = point
                    left = min(left, x)
                    right = max(right, x)
                    top = min(top, y)
                    bottom = max(bottom, y)

                # Calculate width and height
                width = right - left
                height = bottom - top
                
                # Calculate normalized values
                x_center = (left + width / 2) / img_width
                y_center = (top + height / 2) / img_height
                width /= img_width
                height /= img_height
                
                # Write to the destination annotation file
                fp.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("Conversion completed successfully!")


Conversion completed successfully!


In [9]:
pip install ultralytics


Note: you may need to restart the kernel to use updated packages.


In [10]:
!pip install wandb


In [11]:
# Import necessary libraries
from ultralytics import YOLO
import os
import wandb

# Initialize Weights and Biases (W&B) for training phase
wandb.init(project="Caries_Detector", name="YOLOv8_Training_Experiment")

# Log hyperparameters to W&B
wandb.config.update({
    "epochs": 20,
    "batch_size": 16,
    "img_size": 640,
    "model_type": "YOLOv8m"
})


wandb: Currently logged in as: sanikajumde545 (sanikajumde545-symbiosis-international) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Import necessary libraries
from ultralytics import YOLO
import os
import wandb

# Step 1: Initialize Weights and Biases (W&B) for experiment tracking
wandb.init(project="Caries_Detector", name="YOLOv8_Training_Experiment")

# Log hyperparameters and classes to W&B
wandb.config.update({
    "epochs": 20,
    "batch_size": 16,
    "img_size": 640,
    "model_type": "YOLOv8m",
    "classes": ["tooth detection", "caries detection", "dental crack", "dental cavity"]
})

# Step 2: Load the YOLOv8 model
model = YOLO("yolov8m.pt")  # Load the YOLOv8 medium model

# Path to dataset YAML file
data_yaml_path = os.path.join(os.getcwd(), '/Users/sanikajumde/Downloads/yolo_dataset/data.yaml')  # Update to your YAML file path

# Step 3: Train the Model
model.train(
    data=data_yaml_path,          # Path to the dataset YAML file
    epochs=wandb.config.epochs,   # Number of epochs
    batch=wandb.config.batch_size,  # Batch size
    imgsz=wandb.config.img_size,    # Input image size
    project="Caries_Detector",      # W&B project name
    name="YOLOv8_Training",         # Experiment name
    save_period=1                   # Save model weights every epoch
)

# Step 4: Save the Best Model Weights
best_weights_path = "runs/train/YOLOv8_Training/weights/best.pt"  # Path to best weights
wandb.save(best_weights_path)

# Save the trained model explicitly
trained_model_path = "yolov8_trained_model.pt"
model.save(trained_model_path)
wandb.log({"trained_model_path": trained_model_path})

# Step 5: Finish W&B Logging
wandb.finish()


wandb: Currently logged in as: sanikajumde545 (sanikajumde545-symbiosis-international) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Ultralytics 8.3.223 🚀 Python-3.13.5 torch-2.9.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/sanikajumde/Downloads/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=YOLOv8_Training5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots

In [ ]:
def test_model(model_path, image_paths):
    import matplotlib.pyplot as plt
    from PIL import Image, ImageDraw

    # Load the trained model
    model = YOLO(model_path)

    # Prepare visualization
    fig, axs = plt.subplots(len(image_paths), 1, figsize=(10, 5 * len(image_paths)))

    if len(image_paths) == 1:
        axs = [axs]  # Ensure axs is iterable if only one image is provided

    for i, image_path in enumerate(image_paths):
        # Load image
        img = Image.open(image_path)

        # Perform detection
        results = model(image_path)

        # Extract bounding boxes and labels
        boxes = results[0].boxes
        detected_boxes = []
        for box in boxes:
            class_id = int(box.cls.cpu().numpy())  # Class index
            class_name = results[0].names[class_id]  # Class name
            confidence = float(box.conf.cpu().numpy())  # Confidence score
            box_coords = box.xyxy.cpu().numpy()[0]  # Ensure correct array indexing
            
            # Convert coordinates to integers
            box_coords = [int(coord) for coord in box_coords]
            
            # Append the detection
            detected_boxes.append((class_name, confidence, box_coords))

        # Draw bounding boxes on image
        draw = ImageDraw.Draw(img)
        for class_name, confidence, box_coords in detected_boxes:
            draw.rectangle(box_coords, outline="#00FF00", width=3)
            draw.text(
                (box_coords[0], box_coords[1] - 10), 
                f"{class_name}: {confidence:.2f}", 
                fill="white"
            )

        # Plot results
        axs[i].imshow(img)
        axs[i].set_title(f"Detections for {os.path.basename(image_path)}")
        axs[i].axis('off')

    plt.tight_layout()
    plt.show()


# Specify image paths
image_paths = [
    "/Users/sanikajumde/Downloads/yolo_dataset/test/images/1047_jpg.rf.b989b65e4e86d4738757caa25e11fd86.jpg",
    "/Users/sanikajumde/Downloads/yolo_dataset/test/images/2915_jpg.rf.c3f888a3734813d637782191c8e44c4c.jpg",
    "/Users/sanikajumde/Downloads/yolo_dataset/test/images/2642_jpg.rf.39a27997f01d2f33c63986bf2fe47679.jpg"
]

# Test the model
test_model("/Users/sanikajumde/Downloads/yolov8_trained_model.pt", image_paths)


In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Load the trained model
model = YOLO("/Users/sanikajumde/Downloads/yolov8_trained_model.pt")  # Path to your trained model weights

def preprocess_image(image_path, target_size=(640, 640)):
    """
    Preprocess the input image to match YOLOv8's requirements.
    Resize and normalize the image.
    """
    img = Image.open(image_path).convert("RGB")  # Convert to RGB
    img_resized = img.resize(target_size)       # Resize to target size
    return img_resized

def predict_and_visualize(model, image_path):
    """
    Perform prediction on an external image and visualize the results.
    """
    # Open the original image
    img = Image.open(image_path).convert("RGB")

    # Perform inference
    results = model.predict(source=image_path, conf=0.5)  # Adjust confidence threshold as needed
    
    # Extract predictions
    detections = results[0]
    detected_boxes = []
    for box in detections.boxes:
        class_id = int(box.cls.cpu().numpy())  # Class index
        class_name = results[0].names[class_id]  # Class name
        confidence = float(box.conf.cpu().numpy())  # Confidence score
        
        # Bounding box coordinates (x_min, y_min, x_max, y_max)
        box_coords = box.xyxy.cpu().numpy().flatten()  # Flatten the tensor to extract coordinates
        box_coords = [int(coord) for coord in box_coords]  # Convert to integers
        detected_boxes.append((class_name, confidence, box_coords))
    
    # Draw bounding boxes on the image
    draw = ImageDraw.Draw(img)
    for class_name, confidence, box_coords in detected_boxes:
        draw.rectangle(box_coords, outline="#00FF00", width=3)  # Green box
        draw.text((box_coords[0], box_coords[1] - 10), f"{class_name}: {confidence:.2f}", fill="yellow")

    # Display the image with predictions
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Detection Results")
    plt.show()

# Path to the external image
external_image_path = "/Users/sanikajumde/Downloads/yolo_dataset/test/images/1048_jpg.rf.20a105a4c994762333b244fabb7e7fd4.jpg/"

# Run the prediction and visualize the results
predict_and_visualize(model, external_image_path)
